In [ ]:
import sys
import pickle
from pathlib import Path

import pandas as pd
import geopandas as gpd 
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt

import config, data, models, train, evaluate

In [ ]:
save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

subbasins = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'uparea_km2', 'is_gauge', 'outlet_id'])
subbasins.index = subbasins.index.astype(str)

In [ ]:
exp_root = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn")
target = 'discharge'

exp_dirs = [p.parent for p in (exp_root).rglob('results/test_results.parquet')] # dirs with test_data.pkl
exp_dirs

In [ ]:
exp_root =  Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn_spq")
target = 'discharge'

exp_dirs = [
    "/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn/era5_20260116_145516",
    "/nas/cee-water/cjgleason/ted/swot-ml/runs/sf_grnn/swot_minimal_20260116_145814",
]
exp_dirs = [Path(d) / 'results' for d in exp_dirs]
exp_dirs

In [ ]:
def calc_discharge(df, subs):
    uparea = subs['uparea_km2'] 
    mapped_uparea = df.index.get_level_values('subbasin').map(uparea)
    
    df[('pred', 'discharge')] = df[('pred','unit_discharge')] * mapped_uparea
    df[('obs', 'discharge')] = df[('obs','unit_discharge')] * mapped_uparea
    return df

In [ ]:
fig_dir = exp_root / "_figures"
fig_dir.mkdir(exist_ok=True, parents=True)

from tqdm import tqdm
import evaluate

exps = []
for exp_dir in tqdm(exp_dirs):
    try:
        exp_name = ' '.join(exp_dir.parent.stem.split('_')[:-2])
        
        results = pd.read_parquet(exp_dir / 'test_results.parquet').dropna()

        if 'unit_discharge' in results['pred'].columns:
            results = calc_discharge(results, subbasins)

        bulk_m = evaluate.get_all_metrics(results)
        basin_m = evaluate.get_basin_metrics(results)

        # with open(exp_dir / "test_metrics.pkl", 'rb') as f:
        #     bulk_m, basin_m = pickle.load(f)
        
        exps.append((exp_name, results, bulk_m, basin_m))
    except Exception as e:
        print(f"{exp_name}: {e}")
        pass

exps = sorted(exps, key=lambda x: x[0])

In [ ]:
exps[0] = ('ERA5',) + exps[0][1:]
exps[1] = ('ERA5 + SWOT',) + exps[1][1:]

In [ ]:
plt.close('all')

target = 'discharge'
num_models = len(exps)

metric_names = {
    'KGE': 'KGE',
    'corr': '(r - 1)²',
    'alpha': '(alpha - 1)²',
    'beta': '(beta - 1)²'
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes = axes.flatten()

for ax, (name, title) in zip(axes, metric_names.items()):
    for exp_name, results, bulk_metrics, basin_metrics in exps:
        x = np.array(basin_metrics[target][name], dtype=float)
        
        if name in ['alpha', 'beta', 'corr']:
            x = (x-1)**2
        
        
        x = x[~np.isnan(x)]
        ax.ecdf(x, linewidth=1.25, label=exp_name)
    
    ax.set_title(f"{title}")

axes[0].set_xlim([-1, 1])
axes[1].set_xlim([0, 1])
axes[2].set_xlim([0, 1])
axes[3].set_xlim([0, 1])

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.04, 0.82))

# fig.subplots_adjust(left=0.1, right=0.95, bottom=0.2, top=0.95, wspace=0.3, hspace=0.4)
plt.suptitle('SWOT Data Fusion')
plt.tight_layout()
plt.show()

# fig.savefig(fig_dir / f"KGE_decomp.png", dpi=300)

In [ ]:
import xarray as xr

basin_mask = (subbasins['outlet_id'] == 'USGS-07374000')
test_sub_mask = subbasins.index.isin(results.index.get_level_values('subbasin').unique())
basin_gauges = subbasins[basin_mask & test_sub_mask].index
basin_ds = xr.open_zarr("/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched/USGS-07374000")
gauge_ds = basin_ds.sel(subbasin=basin_gauges)


# ds = ds.sel(subbasin=reach_id, date=slice(start_date, end_date))
# has_swot_obs = ~np.isnan(ds['d_wse_river'].to_numpy())

In [ ]:
gauge_swot_count = (~gauge_ds['d_wse_river'].to_dataframe().isna()).groupby('subbasin').apply(np.sum, axis=0)
gauge_swot_count.sort_values(by='d_wse_river', inplace=True)

In [ ]:
gauge_swot_count

In [ ]:
joined_kge = exps[0][3][target]['KGE'].rename('era5').to_frame().join(exps[1][3][target]['KGE'].rename('swot'))
joined_kge['d_kge'] = joined_kge['swot'] - joined_kge['era5'] 
joined_kge = joined_kge.join(gauge_swot_count.rename(columns={'d_wse_river':'num_swot_obs'}))
joined_kge = joined_kge[joined_kge['num_swot_obs']>0]

joined_kge.plot.scatter('d_kge','num_swot_obs')
plt.xlim([-0.25, 1])

In [ ]:
kge = exps[1][3][target]['KGE'].dropna().sort_values(ascending=False)

kge[kge>0]

In [ ]:
joined_kge

In [ ]:
has_swot_obs.sum()

In [ ]:
import xarray as xr

reach_id = 'EAUF-J3631810'
start_date = "2024-01-01"
end_date = "2024-12-31"
metric = 'KGE'

# Horrible hackishness
basin_id = subbasins.loc[reach_id]['outlet_id']
basin_ds = xr.open_zarr(f"/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched/{basin_id}")
subbasin_ds = basin_ds.sel(subbasin=reach_id, date=slice(start_date, end_date))
has_swot_obs = ~np.isnan(subbasin_ds['d_wse_river'].to_numpy())

print(np.sum(has_swot_obs))

fig, ax = plt.subplots(1, 1, figsize=(8, 3))

for exp_name, results, bulk_metrics, basin_metrics in exps: 
    xs = results.xs(reach_id, level='subbasin').sort_index()
    xs = xs.droplevel(0,0)
    xs = xs.loc[start_date:end_date]

    x = xs.index
    y = xs['pred'][target]
    plt.plot(x, y, label=exp_name)
    
    print(f"{exp_name}: {basin_metrics.loc[reach_id][target][metric]:0.2f}")
    
ax.plot(xs.index, xs['obs'][target], color='black', zorder=0, label='gauge')

plt.legend()
plt.title(reach_id)
plt.ylabel("Discharge (m³/s")
plt.show()
# fig.savefig(fig_dir / f"timeseries_{reach_id}.png", dpi=300)

In [ ]:
gauge_swot_count = gauge_swot_count.sort_values(by='d_wse_river', ascending=False)

In [ ]:

reach_ds = ds.sel(subbasin=reach_id, date=slice(start_date, end_date))
date = reach_ds.date.values
d_wse = reach_ds['d_wse_river'].values
# plt.scatter(date,d_wse)

In [ ]:
kge = exps[1][3][target]['KGE'].dropna().sort_values(ascending=False)

# kge[kge>0]

In [ ]:
row

In [ ]:
reach_id

In [ ]:
basin_id

In [ ]:
metric = 'KGE'
target='discharge'
start_date = '2024-01-01'
end_date = '2024-12-31'
colors = ['tab:blue','tab:orange']

zbs_root = Path("/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched/")

for reach_id, row in joined_kge.sort_values('num_swot_obs', ascending=False).iloc[:20].iterrows():

    basin_id = subbasins.loc[reach_id]['outlet_id']
    basin_ds = xr.open_zarr(zbs_root / basin_id)
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    print(f"{reach_id}: {row['num_swot_obs']:.0f}")

    for (exp_name, results, bulk_metrics, basin_metrics), color in zip(exps, colors): 
        x = results.xs(reach_id, level='subbasin').sort_index()
        x = x.droplevel(0,0)
        x = x.loc[start_date:end_date]
        x['pred'][target].plot(ax=ax, color = color, label=exp_name)
        print(f"{exp_name}: {basin_metrics.loc[reach_id][target][metric]:0.2f}")

    x['obs'][target].plot(color='black', ax=ax, zorder=0, label='gauge')

    ax2 = ax.twinx()  # shares x axis but has its own y axis
    
    reach_ds = basin_ds.sel(subbasin=reach_id, date=slice(start_date, end_date))
    date = reach_ds.date.values
    d_wse = reach_ds['d_wse_river'].values / 50
    ax2.scatter(date, d_wse, color='black', s=20, zorder=10, label='d_wse')
        
    
    # plt.legend()
    plt.title(reach_id)
    plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))

exp_idx = 1
name = exps[exp_idx][0]

x = exps[exp_idx][1]['obs'][target]
y = exps[exp_idx][1]['pred'][target]
positive_mask = (x > 0) & (y > 0)
x = x[positive_mask]
y = y[positive_mask]

min_val = 5E-2
max_val = 5E4
log_min = np.log10(min_val)
log_max = np.log10(max_val)

hb = ax.hexbin(x, y, gridsize=(50,50), bins='log', mincnt=5,
            linewidth=0.1,
            extent=(log_min, log_max, log_min, log_max),
            xscale='log', yscale='log')
plt.colorbar(hb, shrink=0.6, aspect=10, anchor=(0,0.5))

# Add a 1:1 line over the min and max of x and y
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)

# Setting axes to be square and equal range
ax.axis('square')
ax.set_xlim(min_val, max_val)
ax.set_ylim(min_val, max_val)
ax.set_title(f"{name}: Discharge (n {len(x):,})")
ax.set_xlabel(f'Observed [m3/s]')
ax.set_ylabel(f'Predicted [m3/s]')
plt.show()

# fig.savefig(fig_dir / f"{name}_bulk_scatterplots.png", dpi=300)